In [1]:
!pip install langchain langchain-community langchain-groq chromadb \ sentence-transformers pypdf -q

In [2]:
!pip install -q \
langchain==0.2.17 \
langchain-community==0.2.19 \
langchain-groq==0.1.10 \
chromadb==0.5.5 \
sentence-transformers==2.7.0 \
pypdf

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

print("All imports successful!")

All imports successful!


In [4]:
!pip install langchain langchain-community langchain-groq chromadb \ sentence-transformers pypdf -q

In [5]:
from langchain_groq import ChatGroq
llm = ChatGroq(api_key="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", model="llama-3.1-8b-instant", temperature=0)

In [6]:
from google.colab import files
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_core.documents import Document

In [7]:
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Uploaded File:", pdf_file)

Saving Strategic Implementation Plan for an MCP.pdf to Strategic Implementation Plan for an MCP (3).pdf
Uploaded File: Strategic Implementation Plan for an MCP (3).pdf


In [8]:
reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    if page.extract_text():
        text += page.extract_text()

print("Total Characters:", len(text))
print("\nFirst 500 Characters:\n")
print(text[:500])

Total Characters: 1793

First 500 Characters:

Strategic Implementation Plan for an MCP-Powered Enterprise Agent System (Summary) 
To solve OrionOps' scalability, reliability, and governance challenges, the proposed solution is a 
modular MCP-powered agent platform with secure tool integration, intelligent workflows, and 
strong monitoring. 
• Standardized MCP Architecture: Use MCP servers with stdio (local) and HTTP (distributed) 
transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose 
enterprise data as se


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

print("Number of Chunks:", len(chunks))
print("\nFirst Chunk:\n")
print(chunks[0])

Number of Chunks: 5

First Chunk:

Strategic Implementation Plan for an MCP-Powered Enterprise Agent System (Summary) 
To solve OrionOps' scalability, reliability, and governance challenges, the proposed solution is a 
modular MCP-powered agent platform with secure tool integration, intelligent workflows, and 
strong monitoring. 
• Standardized MCP Architecture: Use MCP servers with stdio (local) and HTTP (distributed) 
transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose


In [11]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Convert string chunks to Document objects
document_chunks = [Document(page_content=chunk) for chunk in chunks]

db = Chroma.from_documents(document_chunks, embeddings)

/tmp/ipykernel_12474/1232390182.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
ERROR:chromadb.teleme

In [12]:
db = Chroma.from_documents(document_chunks, embeddings)

retriever = db.as_retriever()

docs = retriever.invoke("What is AI?")
print(docs)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(page_content='transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose \nenterprise data as secure, read-only URI-based MCP resources. \n• Smart Discovery and Routing: Agents automatically discover available MCP servers and their \ncapabilities. Requests are routed dynamically to the most suitable tools or servers based on \nuser intent, making the system flexible and easy to expand. \n• Reflexive Reasoning and Self-Healing: Implement an actor–critic model where one agent'), Document(page_content='transports. Organize tools using namespaces (e.g., documents.*, operations.*) and expose \nenterprise data as secure, read-only URI-based MCP resources. \n• Smart Discovery and Routing: Agents automatically discover available MCP servers and their \ncapabilities. Requests are routed dynamically to the most suitable tools or servers based on \nuser intent, making the system flexible and easy to expand. \n• Reflexive Reasoning and Self-Healing: Implemen

In [13]:
def answer_from_pdf(query):
    docs = db.as_retriever().invoke(query)
    context = '\n'.join(d.page_content for d in docs)
    prompt = (
        f"Use ONLY this context to answer.\n{context}\n\n"
        f"Question: {query}\n"
        "If the context does not contain the answer, reply exactly: "
        "'I don't know'."
    )
    return llm.invoke(prompt).content

In [14]:
def adaptive_answer(question, max_tries=3):
    query = question
    for attempt in range(1, max_tries + 1):
        print(f'Attempt {attempt} with query: {query}')
        answer = answer_from_pdf(query)
        if "i don't know" not in answer.lower():
            return answer # good answer, stop
        # otherwise, rephrase and retry
        query = llm.invoke(
            f'Rephrase this search query differently: {query}').content
    return 'Could not find an answer after several tries.'

print(adaptive_answer('What is the main conclusion of the document?'))

Attempt 1 with query: What is the main conclusion of the document?
The main conclusion of the document is that this architecture creates a secure, scalable, reliable, and explainable enterprise agent.
